# Fire Hazard Environment Check - Detection and Segmentation
Run this in Google Colab before training. It validates your current YOLO polygon dataset as segmentation, optionally creates a separate detection dataset from polygon bounding extents, validates both, checks GPU/packages, previews labels, and writes a final report. It does not train any model.

## 1. Project Configuration

In [ ]:
RUN_MODE = 'both'  # 'segmentation', 'detection', or 'both'
AUTO_CREATE_DETECTION_DATASET = True
OVERWRITE_DETECTION_DATASET = False

SEG_DATASET_ROOT = '/content/drive/MyDrive/fire_hazard_dataset/Fire Hazard YOLO26'
DET_DATASET_ROOT = '/content/drive/MyDrive/fire_hazard_dataset/Fire Hazard YOLO26 Detection'
OUTPUT_ROOT = '/content/drive/MyDrive/fire_hazard_experiment_outputs'

SEG_DATA_YAML = f'{SEG_DATASET_ROOT}/data.yaml'
DET_DATA_YAML = f'{DET_DATASET_ROOT}/data.yaml'
EXPECTED_IMAGE_COUNT = 23
seed = 42
checks = {}
reports = {}
print('RUN_MODE:', RUN_MODE)
print('SEG_DATASET_ROOT:', SEG_DATASET_ROOT)
print('DET_DATASET_ROOT:', DET_DATASET_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

## 2. Mount Google Drive

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
SEG_DATASET_ROOT_PATH = Path(SEG_DATASET_ROOT)
DET_DATASET_ROOT_PATH = Path(DET_DATASET_ROOT)
OUTPUT_ROOT_PATH = Path(OUTPUT_ROOT)
ENV_CHECK_ROOT = OUTPUT_ROOT_PATH / 'environment_check'
if SEG_DATASET_ROOT_PATH.exists():
    print('Segmentation dataset found:', SEG_DATASET_ROOT)
    checks['seg_dataset_root'] = True
else:
    print('Segmentation dataset directory not found.\nExpected:')
    print(SEG_DATASET_ROOT)
    checks['seg_dataset_root'] = False
print('Detection dataset exists:', DET_DATASET_ROOT_PATH.exists())

## 3. Clone or Locate Repository

In [ ]:
import subprocess, sys
from pathlib import Path
REPO_URL = 'https://github.com/rachataktn/fire-hazard-spatial.git'
REPO_ROOT = Path('/content/fire-hazard-spatial')
if not (REPO_ROOT / 'scripts' / 'check_dataset.py').exists():
    try:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
    except Exception as exc:
        print('Repository clone failed. If the repo is private, make it public or clone with a read-only token.')
        print(exc)
if (REPO_ROOT / 'scripts' / 'check_dataset.py').exists():
    checks['repository'] = True
    print('Repository ready:', REPO_ROOT)
else:
    checks['repository'] = False
    print('Repository script missing:', REPO_ROOT / 'scripts' / 'check_dataset.py')

## 4. GPU and Runtime Check

In [ ]:
import platform, sys
print('Python:', sys.version)
print('Operating system:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    print('CUDA version:', torch.version.cuda)
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2))
        checks['gpu'] = True
    else:
        print('\nWARNING: CUDA GPU is not available.\nBefore training:\nRuntime -> Change runtime type -> GPU')
        checks['gpu'] = False
except Exception as exc:
    print('PyTorch check failed:', exc)
    checks['gpu'] = False

## 5. Install/Check Packages

In [ ]:
import importlib, subprocess, sys
packages = {'ultralytics':'ultralytics', 'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib', 'Pillow':'PIL', 'PyYAML':'yaml', 'opencv-python-headless':'cv2'}
missing = []
for package, import_name in packages.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        missing.append(package)
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('All required packages are already importable.')
import ultralytics, numpy as np, pandas as pd
print('Ultralytics version:', ultralytics.__version__)
print('NumPy version:', np.__version__)
print('Pandas version:', pd.__version__)
checks['packages'] = True

## 6. Inspect YOLO Model Availability

In [ ]:
from pathlib import Path
import ultralytics
root = Path(ultralytics.__file__).resolve().parent
def package_mentions(term):
    hits = []
    for path in root.rglob('*.py'):
        try:
            text = path.read_text(encoding='utf-8', errors='ignore')
        except Exception:
            continue
        if term.lower() in text.lower():
            hits.append(str(path.relative_to(root)))
    return hits[:25]
mentions_yolo26 = package_mentions('yolo26')
mentions_segment = package_mentions('segment')
mentions_depth = package_mentions('depth')
checks['yolo26_detection'] = bool(mentions_yolo26)
checks['yolo26_segmentation'] = bool(mentions_yolo26) and bool(mentions_segment)
checks['yolo26_depth'] = bool(mentions_yolo26) and bool(mentions_depth)
print('YOLO26 detection available:', 'YES' if checks['yolo26_detection'] else 'NO / not found in installed package text')
print('YOLO26 segmentation available:', 'YES' if checks['yolo26_segmentation'] else 'NO / requires official API verification')
print('YOLO26-Depth available:', 'YES' if checks['yolo26_depth'] else 'NO / requires official API verification')
print('Candidate detection checkpoints: yolo26n.pt, yolo26s.pt, yolo26m.pt, yolo26l.pt, yolo26x.pt')
print('Candidate segmentation checkpoints: yolo26n-seg.pt, yolo26s-seg.pt, yolo26m-seg.pt, yolo26l-seg.pt, yolo26x-seg.pt')
print('Candidate depth checkpoints: yolo26n-depth.pt or official equivalent')
print('Depth output units: metric/relative requires official runtime verification; do not infer from colors.')

## 7. Inspect Dataset Folders

In [ ]:
def print_tree(root, max_depth=4, max_entries=220):
    root = Path(root)
    print(root.name + '/')
    count = 0
    for path in sorted(root.rglob('*')):
        rel = path.relative_to(root)
        depth = len(rel.parts)
        if depth > max_depth:
            continue
        count += 1
        if count > max_entries:
            print('... output truncated ...')
            break
        print('    ' * (depth - 1) + '+-- ' + path.name + ('/' if path.is_dir() else ''))
if SEG_DATASET_ROOT_PATH.exists():
    print('Segmentation dataset:')
    print_tree(SEG_DATASET_ROOT_PATH)
    print('\nsegmentation data.yaml contents:')
    print(Path(SEG_DATA_YAML).read_text(encoding='utf-8') if Path(SEG_DATA_YAML).exists() else 'missing')
if DET_DATASET_ROOT_PATH.exists():
    print('\nDetection dataset:')
    print_tree(DET_DATASET_ROOT_PATH)
    print('\ndetection data.yaml contents:')
    print(Path(DET_DATA_YAML).read_text(encoding='utf-8') if Path(DET_DATA_YAML).exists() else 'missing')

## 8. Optionally Create Detection Dataset from Segmentation Polygons

In [ ]:
if RUN_MODE in {'detection', 'both'} and AUTO_CREATE_DETECTION_DATASET and checks.get('repository'):
    cmd = [sys.executable, str(REPO_ROOT / 'scripts' / 'check_dataset.py'), '--dataset-root', SEG_DATASET_ROOT, '--data-yaml', SEG_DATA_YAML, '--task-mode', 'segmentation', '--expected-images', str(EXPECTED_IMAGE_COUNT), '--convert-detection-root', DET_DATASET_ROOT]
    if OVERWRITE_DETECTION_DATASET:
        cmd.append('--overwrite-converted')
    result = subprocess.run(cmd, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print('STDERR:', result.stderr)
    DET_DATASET_ROOT_PATH = Path(DET_DATASET_ROOT)
else:
    print('Detection dataset conversion skipped.')

## 9. Validate Segmentation and Detection Datasets

In [ ]:
import json
def run_validator(name, root, yaml_path, task_mode):
    report_json = Path(f'/tmp/{name}_dataset_report.json')
    cmd = [sys.executable, str(REPO_ROOT / 'scripts' / 'check_dataset.py'), '--dataset-root', root, '--data-yaml', yaml_path, '--task-mode', task_mode, '--expected-images', str(EXPECTED_IMAGE_COUNT), '--json-output', str(report_json)]
    result = subprocess.run(cmd, text=True, capture_output=True)
    print('\n' + '=' * 60)
    print(name.upper(), task_mode.upper(), 'VALIDATION')
    print('=' * 60)
    print(result.stdout)
    if result.stderr:
        print('STDERR:', result.stderr)
    report = json.loads(report_json.read_text(encoding='utf-8')) if report_json.exists() else {'status': 'INVALID'}
    reports[name] = report
    checks[f'{name}_valid'] = report.get('status') == 'VALID'
    return report
if not checks.get('repository'):
    print('Validation skipped because repository script is unavailable.')
elif RUN_MODE == 'segmentation':
    run_validator('segmentation', SEG_DATASET_ROOT, SEG_DATA_YAML, 'segmentation')
elif RUN_MODE == 'detection':
    run_validator('detection', DET_DATASET_ROOT, DET_DATA_YAML, 'detection')
elif RUN_MODE == 'both':
    run_validator('segmentation', SEG_DATASET_ROOT, SEG_DATA_YAML, 'segmentation')
    run_validator('detection', DET_DATASET_ROOT, DET_DATA_YAML, 'detection')
else:
    raise ValueError(f'Unsupported RUN_MODE: {RUN_MODE}')

## 10. Test Writing Outputs to Google Drive

In [ ]:
try:
    ENV_CHECK_ROOT.mkdir(parents=True, exist_ok=True)
    test_file = ENV_CHECK_ROOT / 'environment_test.txt'
    expected = 'Google Drive output write test successful.'
    test_file.write_text(expected, encoding='utf-8')
    checks['drive_output'] = test_file.read_text(encoding='utf-8') == expected
    print('Google Drive output write test:', 'PASS' if checks['drive_output'] else 'FAIL')
    print('Test file:', test_file)
except Exception as exc:
    checks['drive_output'] = False
    print('Google Drive output write test: FAIL', exc)

## 11. Visual Annotation Preview

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
def label_for_image(image_path):
    parts = list(image_path.parts)
    for i, part in enumerate(parts):
        if part == 'images':
            parts[i] = 'labels'
            return Path(*parts).with_suffix('.txt')
    return image_path.with_suffix('.txt')
def draw_preview(dataset_root, report, task_mode, out_dir):
    dataset_root = Path(dataset_root)
    out_dir.mkdir(parents=True, exist_ok=True)
    class_names = {int(k): v for k, v in report.get('classes', {}).items()} if report else {}
    images = sorted([p for p in dataset_root.rglob('*') if p.suffix.lower() in image_exts]) if dataset_root.exists() else []
    random.seed(seed)
    sample_images = random.sample(images, min(6, len(images))) if images else []
    for idx, image_path in enumerate(sample_images, 1):
        im = Image.open(image_path).convert('RGB')
        w, h = im.size
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(im); ax.axis('off')
        label_path = label_for_image(image_path)
        if label_path.exists():
            for raw in label_path.read_text(encoding='utf-8', errors='replace').splitlines():
                vals = raw.split()
                if task_mode == 'detection' and len(vals) == 5:
                    cid, x, y, bw, bh = map(float, vals)
                    x1, y1 = (x - bw / 2) * w, (y - bh / 2) * h
                    ax.add_patch(patches.Rectangle((x1, y1), bw * w, bh * h, fill=False, linewidth=2, edgecolor='lime'))
                    ax.text(x1, max(0, y1 - 4), class_names.get(int(cid), str(int(cid))), color='black', backgroundcolor='lime', fontsize=9)
                elif task_mode == 'segmentation' and len(vals) >= 7 and (len(vals) - 1) % 2 == 0:
                    cid = int(float(vals[0])); coords = list(map(float, vals[1:]))
                    points = [(coords[i] * w, coords[i + 1] * h) for i in range(0, len(coords), 2)]
                    ax.add_patch(patches.Polygon(points, fill=False, linewidth=2, edgecolor='cyan'))
                    ax.text(points[0][0], max(0, points[0][1] - 4), class_names.get(cid, str(cid)), color='black', backgroundcolor='cyan', fontsize=9)
        fig.savefig(out_dir / f'preview_{idx:02d}.png', bbox_inches='tight', dpi=150)
        plt.show(); plt.close(fig)
    return bool(sample_images)
if 'segmentation' in reports:
    checks['segmentation_preview'] = draw_preview(SEG_DATASET_ROOT, reports['segmentation'], 'segmentation', ENV_CHECK_ROOT / 'annotation_preview_segmentation')
if 'detection' in reports:
    checks['detection_preview'] = draw_preview(DET_DATASET_ROOT, reports['detection'], 'detection', ENV_CHECK_ROOT / 'annotation_preview_detection')

## 12. Optional One-Image Inference Smoke Tests

In [ ]:
RUN_DETECTION_SMOKE_TEST = False
RUN_SEGMENTATION_SMOKE_TEST = False
RUN_DEPTH_SMOKE_TEST = False
DETECTION_MODEL_NAME = 'yolo26n.pt'
SEGMENTATION_MODEL_NAME = 'yolo26n-seg.pt'
DEPTH_MODEL_NAME = 'yolo26n-depth.pt'
checks['detection_inference'] = 'NOT RUN'
checks['segmentation_inference'] = 'NOT RUN'
checks['depth_inference'] = 'NOT RUN'
def first_image(root):
    images = sorted([p for p in Path(root).rglob('*') if p.suffix.lower() in image_exts])
    return images[0] if images else None
if RUN_DETECTION_SMOKE_TEST:
    try:
        from ultralytics import YOLO
        out = ENV_CHECK_ROOT / 'yolo26_detection_test'; out.mkdir(parents=True, exist_ok=True)
        result = YOLO(DETECTION_MODEL_NAME).predict(str(first_image(DET_DATASET_ROOT)), save=True, project=str(out), name='predict', exist_ok=True)[0]
        print('Detection boxes available:', hasattr(result, 'boxes'))
        checks['detection_inference'] = 'PASS'
    except Exception as exc:
        print('Detection smoke test failed:', exc); checks['detection_inference'] = 'FAIL'
if RUN_SEGMENTATION_SMOKE_TEST:
    try:
        from ultralytics import YOLO
        out = ENV_CHECK_ROOT / 'yolo26_segmentation_test'; out.mkdir(parents=True, exist_ok=True)
        result = YOLO(SEGMENTATION_MODEL_NAME).predict(str(first_image(SEG_DATASET_ROOT)), save=True, project=str(out), name='predict', exist_ok=True)[0]
        print('Segmentation masks available:', hasattr(result, 'masks'))
        checks['segmentation_inference'] = 'PASS'
    except Exception as exc:
        print('Segmentation smoke test failed:', exc); checks['segmentation_inference'] = 'FAIL'
if RUN_DEPTH_SMOKE_TEST:
    print('Depth smoke test remains optional until official YOLO26-Depth API/checkpoint syntax is confirmed.')
else:
    print('Smoke tests not run by default. Enable only after checkpoint names are confirmed by the API/official docs.')

## 13. Final Environment Report

In [ ]:
def pf(value): return 'PASS' if value else 'FAIL'
wanted = [RUN_MODE] if RUN_MODE in {'segmentation', 'detection'} else ['segmentation', 'detection']
dataset_ready = all(checks.get(f'{name}_valid', False) for name in wanted)
ready = all([checks.get('repository', False), checks.get('seg_dataset_root', False), checks.get('gpu', False), checks.get('packages', False), checks.get('drive_output', False), dataset_ready])
lines = ['===========================================', 'FIRE HAZARD EXPERIMENT - ENVIRONMENT CHECK', '===========================================', '', f'Run mode: {RUN_MODE}', f'Repository: {pf(checks.get("repository", False))}', f'Google Drive: {pf(Path("/content/drive").exists())}', f'Segmentation dataset root: {pf(checks.get("seg_dataset_root", False))}', f'Detection dataset root: {pf(Path(DET_DATASET_ROOT).exists())}', f'Segmentation validation: {pf(checks.get("segmentation_valid", False)) if "segmentation" in wanted else "NOT RUN"}', f'Detection validation: {pf(checks.get("detection_valid", False)) if "detection" in wanted else "NOT RUN"}', f'GPU: {pf(checks.get("gpu", False))}', f'PyTorch: {torch.__version__ if "torch" in globals() else "unknown"}', f'Ultralytics: {ultralytics.__version__ if "ultralytics" in globals() else "unknown"}', f'YOLO26 detection: {pf(checks.get("yolo26_detection", False))}', f'YOLO26 segmentation: {pf(checks.get("yolo26_segmentation", False))}', f'YOLO26-Depth: {pf(checks.get("yolo26_depth", False))}', f'Google Drive output: {pf(checks.get("drive_output", False))}', f'Single-image detection inference: {checks.get("detection_inference", "NOT RUN")}', f'Single-image segmentation inference: {checks.get("segmentation_inference", "NOT RUN")}', f'Single-image depth inference: {checks.get("depth_inference", "NOT RUN")}', '']
for name, report in reports.items():
    lines += [f'{name.title()} images: {report.get("total_images", "unknown")}', f'{name.title()} labels: {report.get("total_labels", "unknown")}', f'{name.title()} annotations: {report.get("total_annotations", "unknown")}', '']
if not ready:
    lines.append('Fix before full experiment:')
    if not checks.get('repository', False): lines.append('- Make the GitHub repo public or clone it with a read-only token so scripts/check_dataset.py is available.')
    if not checks.get('seg_dataset_root', False): lines.append('- Correct SEG_DATASET_ROOT.')
    if not dataset_ready: lines.append('- Review validation output above for invalid dataset paths or labels.')
    if not checks.get('gpu', False): lines.append('- Enable GPU: Runtime -> Change runtime type -> GPU.')
    if not checks.get('drive_output', False): lines.append('- Check OUTPUT_ROOT permissions and Drive storage.')
    lines.append('')
lines += [f'READY FOR FULL EXPERIMENT: {"YES" if ready else "NO"}', '===========================================']
final_report = '\n'.join(lines)
print(final_report)
ENV_CHECK_ROOT.mkdir(parents=True, exist_ok=True)
(ENV_CHECK_ROOT / 'environment_report.txt').write_text(final_report, encoding='utf-8')
print('\nSaved report to:', ENV_CHECK_ROOT / 'environment_report.txt')